# Lesson 13: principal components of the Treasury curve, the same regimes at k = 2 from two scores as from eleven yields

A day of the Treasury curve has eleven yields, from 1 month to 30 years, all in per cent. The principal components of the 5,165 days from 9 February 2006 to 30 September 2026 are computed from the centred yields, unscaled, since all eleven are in one unit. The fewest components with a cumulative share of the variance of at least 95 per cent are the first two, with 98.9 per cent. The days are grouped by k-means into k = 2 rate regimes, once on the two scores per day and once on all eleven yields. The silhouette on the two scores is largest at k = 2, with 0.694 among k = 2 to 6. The regimes are ordered by the mean 3-month yield of their days, and the regimes of the two runs are paired so that the most days have the same regime in both.

Treasury yields: Federal Reserve Board H.15 via FRED, Federal Reserve Bank of St. Louis. Snapshot 30 September 2026.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.optimize import linear_sum_assignment

curve = pd.read_csv("https://digital-ai-finance.github.io/teaching-msc-data-science/data/2026-09-30-treasury-curve-daily.csv", index_col="date", parse_dates=True)
pca = PCA(svd_solver="full").fit(curve)  # centred, unscaled: all eleven yields in per cent
share, cum = 100 * pca.explained_variance_ratio_, 100 * np.cumsum(pca.explained_variance_ratio_)
n95 = (cum < 95).sum() + 1  # the fewest components with a cumulative share of at least 95 per cent
scores = pca.transform(curve)[:, :n95]
print(f"variance shares of PC1, PC2, PC3: {share[0]:.1f}, {share[1]:.1f}, {share[2]:.1f} per cent")
print(f"components for at least 95 per cent of the variance: {n95}, with {cum[n95 - 1]:.1f} per cent")

plt.plot(range(1, 12), cum, marker="o", color="black", label="cumulative share")
plt.axhline(95, ls="--", color="grey", label="cut-off 95 per cent")
plt.xlabel("number of components")
plt.ylabel("cumulative share of the variance (per cent)")
plt.legend(loc="lower right")
plt.show()

In [ ]:
k = 2  # the largest silhouette on the two scores, 0.694 among k = 2 to 6
on_scores = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(scores)
on_yields = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(curve)
days = curve.DGS3MO.groupby(on_scores).agg(["mean", "size"]).sort_values("mean")["size"]
print(f"low-rate regime {days.iloc[0]:,} days, high-rate regime {days.iloc[-1]:,} days")
table = pd.crosstab(on_scores, on_yields).to_numpy()  # days per pair of regimes of the two runs
r, c = linear_sum_assignment(-table)  # the pairing of the regimes with the most days in common
print(f"same regime from two scores as from eleven yields: {table[r, c].sum():,} of {len(curve):,} days")

Change `k = 2` to `k = 3` and run the cell again. At k = 3 the same regime from two scores as from eleven yields is found on 5,156 of the 5,165 days. The low-rate regime has 2,754 days, the high-rate regime 1,480 days, and the other 931 days are in a transition regime between them.